# Uber Pickups - Identification des Hot-Zones à New York City

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/5/58/Uber_logo_2018.svg/1024px-Uber_logo_2018.svg.png" alt="UBER LOGO" width="30%" />

**Projet de Machine Learning Non-Supervisé** | Bloc 3 - JEDHA Data Science Fullstack

## Sommaire

1. [Introduction et Contexte](#1-introduction)
2. [Configuration de l'environnement](#2-config)
3. [Chargement des données](#3-data)
4. [Analyse Exploratoire (EDA)](#4-eda)
5. [Pipeline de Préparation des Données](#5-pipeline)
6. [Analyse en Composantes Principales (PCA)](#6-pca)
7. [Clustering K-Means](#7-kmeans)
8. [Clustering DBSCAN](#8-dbscan)
9. [Comparaison des Algorithmes](#9-comparison)
10. [Généralisation par Jour de la Semaine](#10-generalization)
11. [Recommandations Business](#11-recommendations)
12. [Conclusion et Perspectives](#12-conclusion)

---
## 1. Introduction et Contexte <a id="1-introduction"></a>

**Uber** est l'une des startups les plus emblématiques au monde. Fondée en 2009, l'entreprise s'est diversifiée depuis le VTC vers la livraison (Uber Eats), le fret et la micro-mobilité. Présente dans environ 70 pays et 900 villes, Uber génère plus de 14 milliards de dollars de revenus.

### 1.1 Problématique business

L'un des principaux points de friction identifiés par Uber est le **temps d'attente des utilisateurs**. Les recherches internes montrent que les clients acceptent d'attendre **5 à 7 minutes maximum** avant d'annuler leur course.

Le constat : les chauffeurs ne sont pas toujours positionnés dans les zones à forte demande. L'équipe data d'Uber souhaite développer une fonctionnalité recommandant les **hot-zones** (zones à forte concentration de pickups) aux chauffeurs, en fonction du jour et de l'heure.

### 1.2 Objectifs du projet

- Identifier les **hot-zones de pickups** à New York City
- Visualiser ces zones sur des **cartes interactives**
- Comparer au moins **deux algorithmes de clustering** (K-Means et DBSCAN)
- Analyser les patterns par **jour de la semaine**
- Formuler des **recommandations opérationnelles** pour Uber

### 1.3 Justification du choix algorithmique : apprentissage non-supervisé

| Critère | Justification |
|---------|---------------|
| **Absence de labels** | Aucune variable cible ne définit les hot-zones. Pas de vérité terrain disponible. |
| **Découverte de structure** | Le clustering découvre des groupes naturels dans les données géographiques. |
| **Nature des données** | Les coordonnées GPS (latitude, longitude) se prêtent au regroupement spatial. |
| **Objectif exploratoire** | On cherche à identifier des patterns, pas à prédire une valeur. |

**Algorithmes retenus :**
- **K-Means** : partitionnement en K groupes, rapide et efficace pour des clusters sphériques
- **DBSCAN** : clustering par densité, adapté aux formes arbitraires et capable de détecter le bruit

---
## 2. Configuration de l'environnement <a id="2-config"></a>

In [1]:
import os
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
print("Environnement configuré avec succès.")

Environnement configuré avec succès.


---
## 3. Chargement des données <a id="3-data"></a>

Les données proviennent du dataset **Uber Trip Data**, contenant les pickups à New York City sur plusieurs mois en 2014.

In [2]:
DATA_URL = (
    "https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/"
    "Machine+Learning+non+Supervis%C3%A9/Projects/uber-trip-data.zip"
)
ZIP_PATH = "uber-trip-data.zip"

if not os.path.exists(ZIP_PATH):
    print("Téléchargement des données en cours...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print("Téléchargement terminé.")
else:
    print("Fichier déjà présent localement.")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    all_csv = [f for f in z.namelist() if f.endswith(".csv")]
    csv_files = [
        f for f in all_csv
        if "uber-raw-data-" in f and "__MACOSX" not in f
    ]
    csv_files.sort()
    print(f"\nFichiers CSV utilisés (courses Uber uniquement) : {csv_files}")
    z.extractall(".")

dfs = []
for f in csv_files:
    temp = pd.read_csv(f, encoding="latin-1")
    dfs.append(temp)
    print(f"  {f} : {temp.shape}")

df = pd.concat(dfs, ignore_index=True)
print(f"\nDataset complet : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")

Fichier déjà présent localement.

Fichiers dans l'archive : ['uber-trip-data/taxi-zone-lookup.csv', '__MACOSX/uber-trip-data/._taxi-zone-lookup.csv', 'uber-trip-data/uber-raw-data-apr14.csv', '__MACOSX/uber-trip-data/._uber-raw-data-apr14.csv', 'uber-trip-data/uber-raw-data-aug14.csv', '__MACOSX/uber-trip-data/._uber-raw-data-aug14.csv', 'uber-trip-data/uber-raw-data-sep14.csv', '__MACOSX/uber-trip-data/._uber-raw-data-sep14.csv', 'uber-trip-data/uber-raw-data-jul14.csv', '__MACOSX/uber-trip-data/._uber-raw-data-jul14.csv', 'uber-trip-data/uber-raw-data-jun14.csv', '__MACOSX/uber-trip-data/._uber-raw-data-jun14.csv', 'uber-trip-data/uber-raw-data-may14.csv', '__MACOSX/uber-trip-data/._uber-raw-data-may14.csv']
  uber-trip-data/taxi-zone-lookup.csv : (265, 3)
  __MACOSX/uber-trip-data/._taxi-zone-lookup.csv : (0, 1)
  uber-trip-data/uber-raw-data-apr14.csv : (564516, 4)
  __MACOSX/uber-trip-data/._uber-raw-data-apr14.csv : (0, 1)
  uber-trip-data/uber-raw-data-aug14.csv : (829275, 4)
  

In [3]:
df.head(10)

,LocationID,Borough,Zone,Unnamed: 0,Date/Time,Lat,Lon,Base
0,1.0,EWR,Newark Airport,NaN,NaN,NaN,NaN,NaN
1,2.0,Queens,Jamaica Bay,NaN,NaN,NaN,NaN,NaN
2,3.0,Bronx,Allerton/Pelham Gardens,NaN,NaN,NaN,NaN,NaN
3,4.0,Manhattan,Alphabet City,NaN,NaN,NaN,NaN,NaN
4,5.0,Staten Island,Arden Heights,NaN,NaN,NaN,NaN,NaN
5,6.0,Staten Island,Arrochar/Fort Wadsworth,NaN,NaN,NaN,NaN,NaN
6,7.0,Queens,Astoria,NaN,NaN,NaN,NaN,NaN
7,8.0,Queens,Astoria Park,NaN,NaN,NaN,NaN,NaN
8,9.0,Queens,Auburndale,NaN,NaN,NaN,NaN,NaN
9,10.0,Queens,Baisley Park,NaN,NaN,NaN,NaN,NaN


---
## 4. Analyse Exploratoire des Données (EDA) <a id="4-eda"></a>

In [4]:
print("=== Informations générales ===")
print(f"Dimensions : {df.shape}")
print(f"\nTypes de données :")
print(df.dtypes)
print(f"\n=== Valeurs manquantes ===")
print(df.isnull().sum())
print(f"\n=== Doublons : {df.duplicated().sum()} ===")
print(f"\n=== Statistiques descriptives ===")
df.describe()

=== Informations générales ===
Dimensions : (4534592, 8)

Types de données :
LocationID    float64
Borough           str
Zone              str
Unnamed: 0     object
Date/Time         str
Lat           float64
Lon           float64
Base              str
dtype: object

=== Valeurs manquantes ===
LocationID    4534327
Borough       4534327
Zone          4534327
Unnamed: 0    4534592
Date/Time         265
Lat               265
Lon               265
Base              265
dtype: int64

=== Doublons : 82581 ===

=== Statistiques descriptives ===


,LocationID,Lat,Lon
count,265.000000,4.534327e+06,4.534327e+06
mean,133.000000,4.073926e+01,-7.397302e+01
std,76.643112,3.994991e-02,5.726670e-02
min,1.000000,3.965690e+01,-7.492900e+01
25%,67.000000,4.072110e+01,-7.399650e+01
50%,133.000000,4.074220e+01,-7.398340e+01
75%,199.000000,4.076100e+01,-7.396530e+01
max,265.000000,4.211660e+01,-7.206660e+01


In [5]:
df["Date/Time"] = pd.to_datetime(df["Date/Time"])
df["hour"] = df["Date/Time"].dt.hour
df["day_of_week"] = df["Date/Time"].dt.dayofweek
df["day_name"] = df["Date/Time"].dt.day_name()
df["month"] = df["Date/Time"].dt.month

DAY_ORDER = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]
df["day_name"] = pd.Categorical(
    df["day_name"], categories=DAY_ORDER, ordered=True
)

print(f"Période couverte : {df['Date/Time'].min()} -> {df['Date/Time'].max()}")
df.head()

Période couverte : 2014-04-01 00:00:00 -> 2014-09-30 22:59:00


,LocationID,Borough,Zone,Unnamed: 0,Date/Time,Lat,Lon,Base,hour,day_of_week,day_name,month
0,1.0,EWR,Newark Airport,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,Queens,Jamaica Bay,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,Bronx,Allerton/Pelham Gardens,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,Manhattan,Alphabet City,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,Staten Island,Arden Heights,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Pickups par heure de la journée",
        "Pickups par jour de la semaine",
        "Pickups par mois",
        "Pickups par base Uber"
    )
)

hourly = df.groupby("hour").size().reset_index(name="count")
fig.add_trace(
    go.Bar(x=hourly["hour"], y=hourly["count"], marker_color="#636EFA"),
    row=1, col=1
)

daily = df.groupby("day_name", observed=True).size().reset_index(name="count")
fig.add_trace(
    go.Bar(x=daily["day_name"], y=daily["count"], marker_color="#EF553B"),
    row=1, col=2
)

monthly = df.groupby("month").size().reset_index(name="count")
fig.add_trace(
    go.Bar(x=monthly["month"], y=monthly["count"], marker_color="#00CC96"),
    row=2, col=1
)

base_counts = df.groupby("Base").size().reset_index(name="count")
fig.add_trace(
    go.Bar(x=base_counts["Base"], y=base_counts["count"], marker_color="#AB63FA"),
    row=2, col=2
)

fig.update_layout(
    height=700, showlegend=False,
    title_text="Distributions temporelles des pickups Uber - NYC"
)
fig.show()

In [ ]:
heatmap_data = df.groupby(
    ["day_name", "hour"], observed=True
).size().reset_index(name="count")
heatmap_pivot = heatmap_data.pivot(
    index="day_name", columns="hour", values="count"
)

fig = px.imshow(
    heatmap_pivot,
    labels=dict(x="Heure", y="Jour", color="Pickups"),
    title="Heatmap des pickups par jour et heure",
    color_continuous_scale="YlOrRd",
    aspect="auto"
)
fig.show()

In [ ]:
sample_map = df.sample(n=min(50_000, len(df)), random_state=42)

fig = px.scatter_mapbox(
    sample_map,
    lat="Lat", lon="Lon",
    color_discrete_sequence=["#636EFA"],
    zoom=10, height=600,
    opacity=0.3,
    title="Carte des pickups Uber - NYC (échantillon 50k)"
)
fig.update_layout(
    mapbox_style="open-street-map",
    margin=dict(l=0, r=0, t=40, b=0)
)
fig.show()

---
## 5. Pipeline de Préparation des Données <a id="5-pipeline"></a>

Le pipeline de préparation comprend :
1. **Filtrage géographique** : suppression des coordonnées hors New York City
2. **Suppression des doublons**
3. **Normalisation** avec `StandardScaler` pour le clustering

In [ ]:
def prepare_data(dataframe):
    """
    Pipeline de préparation des données.

    Étapes :
        1. Filtrage des coordonnées hors NYC
        2. Suppression des doublons

    Parameters
    ----------
    dataframe : pd.DataFrame
        DataFrame avec colonnes Lat, Lon

    Returns
    -------
    pd.DataFrame
        DataFrame nettoyé
    """
    data = dataframe.copy()

    n_before = len(data)
    data = data[
        (data["Lat"].between(40.5, 41.0))
        & (data["Lon"].between(-74.3, -73.7))
    ]
    n_filtered = n_before - len(data)
    print(f"Filtrage géographique : {n_filtered} points supprimés")

    n_dup = data.duplicated().sum()
    data = data.drop_duplicates()
    print(f"Doublons supprimés : {n_dup}")

    return data.reset_index(drop=True)


def get_features(dataframe, columns=None):
    """
    Extrait et normalise les features pour le clustering.

    Parameters
    ----------
    dataframe : pd.DataFrame
    columns : list, optional
        Colonnes à utiliser (défaut : ['Lat', 'Lon'])

    Returns
    -------
    tuple : (X_raw, X_scaled, scaler)
    """
    if columns is None:
        columns = ["Lat", "Lon"]

    X = dataframe[columns].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return X, X_scaled, scaler


df_clean = prepare_data(df)
print(f"\nDataset nettoyé : {df_clean.shape[0]:,} lignes")

---
## 6. Analyse en Composantes Principales (PCA) <a id="6-pca"></a>

Avant le clustering, nous utilisons la **PCA** pour :
- Visualiser la **structure des données** en dimensions réduites
- Identifier les **corrélations** entre features géographiques et temporelles
- Vérifier que les données présentent des groupes naturels

In [ ]:
features_pca = ["Lat", "Lon", "hour", "day_of_week"]
sample_pca = df_clean.sample(
    n=min(50_000, len(df_clean)), random_state=42
)

_, X_pca_scaled, _ = get_features(sample_pca, features_pca)

pca = PCA(n_components=len(features_pca))
X_pca = pca.fit_transform(X_pca_scaled)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[f"PC{i+1}" for i in range(len(explained))],
    y=explained, name="Variance individuelle",
    marker_color="#636EFA"
))
fig.add_trace(go.Scatter(
    x=[f"PC{i+1}" for i in range(len(cumulative))],
    y=cumulative, name="Variance cumulée",
    mode="lines+markers", marker_color="#EF553B"
))
fig.update_layout(
    title="Variance expliquée par composante principale",
    yaxis_title="Ratio de variance expliquée",
    xaxis_title="Composante principale"
)
fig.show()

for i, (ind, cum) in enumerate(zip(explained, cumulative)):
    print(f"PC{i+1} : {ind:.2%} (cumulé : {cum:.2%})")

In [ ]:
pca_df = pd.DataFrame(X_pca[:, :2], columns=["PC1", "PC2"])
pca_df["hour"] = sample_pca["hour"].values

fig = px.scatter(
    pca_df, x="PC1", y="PC2",
    color="hour", color_continuous_scale="Viridis",
    opacity=0.3, height=500,
    title="Projection PCA (2 composantes) colorée par heure de la journée"
)
fig.show()

> **Observation PCA** : La part de variance expliquée par les premières composantes indique une structure forte dans les données. Sans détailler ici les loadings, on reste prudent : le clustering **géographique** (Lat/Lon) reste la piste principale ; les features temporelles (heure, jour) ajoutent une variabilité secondaire utile pour la généralisation.

---
## 7. Clustering K-Means <a id="7-kmeans"></a>

**K-Means** partitionne les données en K groupes en minimisant l'**inertie** (somme des distances intra-cluster au carré). Chaque cluster est représenté par son **centroïde**, qui correspond ici à une hot-zone.

### 7.1 Approche incrémentale : "Start small, grow big"

Conformément aux recommandations du projet, nous commençons par un sous-ensemble restreint (**un jour, une heure**) avant de généraliser.

In [ ]:
df_sample = df_clean[
    (df_clean["day_of_week"] == 0) & (df_clean["hour"] == 17)
].copy()

print(f"Échantillon retenu : Lundi 17h (heure de pointe)")
print(f"Nombre de pickups : {len(df_sample):,}")

X_raw, X_scaled, scaler = get_features(df_sample)

### 7.2 Optimisation du nombre de clusters K

Nous combinons une **lecture visuelle** et une **règle de décision explicite** :

- **Méthode du coude (Elbow)** : la courbe d'inertie suggère des plages de K plausibles (réduction du gain marginal lorsque K augmente).
- **Score Silhouette** : mesure la qualité de séparation des clusters (proche de 1 = bon).

**Choix retenu pour la suite :** contrainte métier d'avoir **au moins cinq hot-zones** distinctes (`K ≥ 5`). Parmi ces valeurs, on sélectionne le **K qui maximise la silhouette** sur l'échantillon Lundi 17h. Le coude sert surtout de **contrôle visuel** ; le critère quantitatif final est la silhouette sous cette contrainte.

In [ ]:
K_RANGE = range(2, 16)
inertias = []
sil_scores = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Méthode du Coude (Elbow)", "Score Silhouette")
)

fig.add_trace(
    go.Scatter(
        x=list(K_RANGE), y=inertias,
        mode="lines+markers", marker_color="#636EFA"
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=list(K_RANGE), y=sil_scores,
        mode="lines+markers", marker_color="#EF553B"
    ),
    row=1, col=2
)

fig.update_xaxes(title_text="K", row=1, col=1)
fig.update_xaxes(title_text="K", row=1, col=2)
fig.update_yaxes(title_text="Inertie (WCSS)", row=1, col=1)
fig.update_yaxes(title_text="Silhouette Score", row=1, col=2)
fig.update_layout(
    height=400, showlegend=False,
    title_text="Optimisation du nombre de clusters K"
)
fig.show()

for k, (ine, sil) in zip(K_RANGE, zip(inertias, sil_scores)):
    print(f"K={k:2d} | Inertie={ine:10.1f} | Silhouette={sil:.4f}")

### 7.3 Amélioration itérative : modèle naïf vs modèle optimisé

Pour illustrer l'intérêt du réglage de K, nous comparons :
- **V1 (naïf)** : K=5 choisi arbitrairement, sans analyse préalable
- **V2 (optimisé)** : K choisi parmi les **K ≥ 5** comme celui qui **maximise la silhouette** (cohérent avec la section 7.2)

In [ ]:
NAIVE_K = 5
km_naive = KMeans(n_clusters=NAIVE_K, random_state=42, n_init=10)
labels_naive = km_naive.fit_predict(X_scaled)
sil_naive = silhouette_score(X_scaled, labels_naive)

print(f"=== K-Means V1 (naïf, K={NAIVE_K}) ===")
print(f"Inertie      : {km_naive.inertia_:.1f}")
print(f"Silhouette   : {sil_naive:.4f}")

# Contrainte métier : au moins 5 zones ; parmi les K >= 5, on maximise la silhouette
MIN_K_BUSINESS = 5
valid = {k: s for k, s in zip(K_RANGE, sil_scores) if k >= MIN_K_BUSINESS}
OPTIMAL_K = max(valid, key=valid.get)

km_optimal = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
labels_optimal = km_optimal.fit_predict(X_scaled)
sil_optimal = silhouette_score(X_scaled, labels_optimal)

print(f"\n=== K-Means V2 (optimisé, K={OPTIMAL_K}) ===")
print(f"Inertie      : {km_optimal.inertia_:.1f}")
print(f"Silhouette   : {sil_optimal:.4f}")

if sil_naive != 0:
    improvement = ((sil_optimal - sil_naive) / abs(sil_naive)) * 100
    print(f"\n-> Amélioration Silhouette V1->V2 : {improvement:+.1f}%")

### 7.4 Visualisation des Hot-zones K-Means

In [ ]:
df_sample["cluster_kmeans"] = labels_optimal

centroids = scaler.inverse_transform(km_optimal.cluster_centers_)
centroids_df = pd.DataFrame(centroids, columns=["Lat", "Lon"])
centroids_df["cluster"] = range(OPTIMAL_K)

fig = px.scatter_mapbox(
    df_sample, lat="Lat", lon="Lon",
    color=df_sample["cluster_kmeans"].astype(str),
    zoom=11, height=600,
    title=f"Hot-zones K-Means (K={OPTIMAL_K}) - Lundi 17h",
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.add_trace(go.Scattermapbox(
    lat=centroids_df["Lat"], lon=centroids_df["Lon"],
    mode="markers",
    marker=dict(size=15, color="red", symbol="star"),
    name="Centroïdes (hot-zones)"
))

fig.update_layout(
    mapbox_style="open-street-map",
    margin=dict(l=0, r=0, t=40, b=0)
)
fig.show()

print("Coordonnées des hot-zones (centroïdes K-Means) :")
centroids_df

---
## 8. Clustering DBSCAN <a id="8-dbscan"></a>

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) regroupe les points par **densité**. Contrairement à K-Means :
- Pas besoin de spécifier K à l'avance
- Détecte les **formes arbitraires**
- Identifie le **bruit** (points isolés, label = -1)

### 8.1 Estimation du paramètre epsilon

**`eps`** (rayon du voisinage) est difficile à fixer à l'avance. Nous utilisons :
- une **courbe des k-plus proches voisins** (distance au k-ème voisin triée) comme **aide visuelle** pour un ordre de grandeur ;
- une **grille de valeurs** (`eps`, `min_samples`) au §8.2, au même titre que les essais systématiques vus en cours, pour retenir la combinaison la plus pertinente.

In [ ]:
nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
distances = np.sort(distances[:, -1])

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=distances, mode="lines", line=dict(color="#636EFA")
))
fig.update_layout(
    title="Distance au 5ème plus proche voisin (estimation de eps)",
    xaxis_title="Points (triés par distance croissante)",
    yaxis_title="Distance au 5ème voisin",
    height=400
)
fig.show()

### 8.2 Grid Search des hyperparamètres

Nous testons plusieurs combinaisons de `eps` et `min_samples` pour trouver la configuration optimale.

In [ ]:
eps_values = [0.1, 0.2, 0.3, 0.5, 0.8]
min_samples_values = [3, 5, 10, 15, 20]

results_db = []
for eps in eps_values:
    for ms in min_samples_values:
        db = DBSCAN(eps=eps, min_samples=ms)
        labels_db = db.fit_predict(X_scaled)
        n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
        n_noise = (labels_db == -1).sum()
        noise_pct = n_noise / len(labels_db) * 100

        if n_clusters >= 2:
            mask = labels_db != -1
            sil = silhouette_score(X_scaled[mask], labels_db[mask])
        else:
            sil = -1

        results_db.append({
            "eps": eps,
            "min_samples": ms,
            "n_clusters": n_clusters,
            "bruit_pct": round(noise_pct, 1),
            "silhouette": round(sil, 4)
        })

results_db_df = pd.DataFrame(results_db)
results_db_valid = results_db_df[results_db_df["silhouette"] > 0].copy()
print(f"Combinaisons valides : {len(results_db_valid)} / {len(results_db_df)}")
results_db_valid.sort_values("silhouette", ascending=False).head(10)

### 8.3 Amélioration itérative : modèle naïf vs optimisé

In [ ]:
NAIVE_EPS, NAIVE_MS = 0.5, 5
db_naive = DBSCAN(eps=NAIVE_EPS, min_samples=NAIVE_MS)
labels_db_naive = db_naive.fit_predict(X_scaled)
n_cl_naive_db = len(set(labels_db_naive)) - (1 if -1 in labels_db_naive else 0)
mask_naive_db = labels_db_naive != -1

if n_cl_naive_db >= 2 and mask_naive_db.sum() > 0:
    sil_db_naive = silhouette_score(
        X_scaled[mask_naive_db], labels_db_naive[mask_naive_db]
    )
else:
    sil_db_naive = -1

print(f"=== DBSCAN V1 (naïf, eps={NAIVE_EPS}, min_samples={NAIVE_MS}) ===")
print(f"Clusters     : {n_cl_naive_db}")
print(f"Bruit        : {(~mask_naive_db).sum()} pts ({(~mask_naive_db).mean()*100:.1f}%)")
print(f"Silhouette   : {sil_db_naive:.4f}")

if len(results_db_valid) > 0:
    best_row = results_db_valid.loc[results_db_valid["silhouette"].idxmax()]
    BEST_EPS = best_row["eps"]
    BEST_MS = int(best_row["min_samples"])
else:
    BEST_EPS, BEST_MS = 0.3, 5

db_optimal = DBSCAN(eps=BEST_EPS, min_samples=BEST_MS)
labels_db_optimal = db_optimal.fit_predict(X_scaled)
n_cl_optimal_db = len(set(labels_db_optimal)) - (1 if -1 in labels_db_optimal else 0)
mask_optimal_db = labels_db_optimal != -1

if n_cl_optimal_db >= 2 and mask_optimal_db.sum() > 0:
    sil_db_optimal = silhouette_score(
        X_scaled[mask_optimal_db], labels_db_optimal[mask_optimal_db]
    )
else:
    sil_db_optimal = -1

print(f"\n=== DBSCAN V2 (optimisé, eps={BEST_EPS}, min_samples={BEST_MS}) ===")
print(f"Clusters     : {n_cl_optimal_db}")
print(f"Bruit        : {(~mask_optimal_db).sum()} pts ({(~mask_optimal_db).mean()*100:.1f}%)")
print(f"Silhouette   : {sil_db_optimal:.4f}")

if sil_db_naive > 0 and sil_db_optimal > 0:
    improv = ((sil_db_optimal - sil_db_naive) / abs(sil_db_naive)) * 100
    print(f"\n-> Amélioration Silhouette V1->V2 : {improv:+.1f}%")

### 8.4 Visualisation des Hot-zones DBSCAN

In [ ]:
df_sample["cluster_dbscan"] = labels_db_optimal

df_db_clusters = df_sample[df_sample["cluster_dbscan"] != -1].copy()
df_db_noise = df_sample[df_sample["cluster_dbscan"] == -1].copy()

fig = px.scatter_mapbox(
    df_db_clusters, lat="Lat", lon="Lon",
    color=df_db_clusters["cluster_dbscan"].astype(str),
    zoom=11, height=600,
    title=f"Hot-zones DBSCAN (eps={BEST_EPS}, min_samples={BEST_MS}) - Lundi 17h",
    color_discrete_sequence=px.colors.qualitative.Set2
)

if len(df_db_noise) > 0:
    fig.add_trace(go.Scattermapbox(
        lat=df_db_noise["Lat"], lon=df_db_noise["Lon"],
        mode="markers",
        marker=dict(size=3, color="gray", opacity=0.3),
        name="Bruit"
    ))

fig.update_layout(
    mapbox_style="open-street-map",
    margin=dict(l=0, r=0, t=40, b=0)
)
fig.show()

print(f"Nombre de clusters : {n_cl_optimal_db}")
print(f"Points classés comme bruit : {len(df_db_noise)} ({len(df_db_noise)/len(df_sample)*100:.1f}%)")

---
## 9. Comparaison des Algorithmes <a id="9-comparison"></a>

Comparons les performances des deux algorithmes optimisés sur le même échantillon (Lundi 17h).

In [ ]:
comparison = pd.DataFrame({
    "Critère": [
        "Nombre de clusters",
        "Silhouette Score",
        "Détection du bruit",
        "Points de bruit",
        "Forme des clusters",
        "Hyperparamètres",
    ],
    "K-Means": [
        str(OPTIMAL_K),
        f"{sil_optimal:.4f}",
        "Non",
        "0",
        "Sphériques (convexes)",
        f"K={OPTIMAL_K}",
    ],
    "DBSCAN": [
        str(n_cl_optimal_db),
        f"{sil_db_optimal:.4f}",
        "Oui",
        str((~mask_optimal_db).sum()),
        "Arbitraires (densité)",
        f"eps={BEST_EPS}, min_samples={BEST_MS}",
    ],
})

comparison

In [ ]:
fig_km = px.scatter_mapbox(
    df_sample, lat="Lat", lon="Lon",
    color=df_sample["cluster_kmeans"].astype(str),
    zoom=11, height=500,
    title=f"K-Means (K={OPTIMAL_K})",
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig_km.update_layout(
    mapbox_style="open-street-map",
    margin=dict(l=0, r=0, t=40, b=0)
)
fig_km.show()

fig_db = px.scatter_mapbox(
    df_db_clusters, lat="Lat", lon="Lon",
    color=df_db_clusters["cluster_dbscan"].astype(str),
    zoom=11, height=500,
    title=f"DBSCAN (eps={BEST_EPS}, min_samples={BEST_MS})",
    color_discrete_sequence=px.colors.qualitative.Set2
)
if len(df_db_noise) > 0:
    fig_db.add_trace(go.Scattermapbox(
        lat=df_db_noise["Lat"], lon=df_db_noise["Lon"],
        mode="markers",
        marker=dict(size=3, color="gray", opacity=0.3),
        name="Bruit"
    ))
fig_db.update_layout(
    mapbox_style="open-street-map",
    margin=dict(l=0, r=0, t=40, b=0)
)
fig_db.show()

### 9.1 Discussion et synthèse

**K-Means :**
- Rapide et simple à interpréter
- Les centroïdes fournissent directement les coordonnées GPS des hot-zones
- Suppose des clusters sphériques (pas toujours réaliste pour des zones urbaines)
- Pas de détection du bruit : tous les points sont assignés à un cluster

**DBSCAN :**
- Détecte les zones denses de forme quelconque
- Identifie les points isolés (bruit) qui ne correspondent pas à des zones récurrentes
- Plus sensible au choix des paramètres `eps` et `min_samples`
- Les "centres" des zones ne sont pas fournis directement

**Pour ce cas d'usage**, K-Means est particulièrement adapté car :
1. Les centroïdes donnent directement les **coordonnées GPS recommandées** aux chauffeurs
2. Le nombre de zones peut être **contrôlé** via K selon les besoins opérationnels
3. DBSCAN complète l'analyse en identifiant les **zones de bruit** à éviter

---
## 10. Généralisation par Jour de la Semaine <a id="10-generalization"></a>

Nous appliquons maintenant le clustering K-Means à **chaque jour de la semaine** pour identifier comment les hot-zones évoluent.

**Simplification assumée :** nous réutilisons le **même K** que celui optimisé sur **Lundi 17h**, et nous agrégons **toutes les heures** du jour (avec sous-échantillonnage pour la performance). Un modèle plus fin recalculerait K par jour ou par créneau horaire ; ici le gain est la **comparabilité** des cartes entre jours.

In [ ]:
def cluster_by_day(dataframe, n_clusters, max_samples=20_000):
    """
    Applique K-Means pour chaque jour de la semaine.

    Parameters
    ----------
    dataframe : pd.DataFrame
    n_clusters : int
    max_samples : int
        Limite de points par jour pour la performance

    Returns
    -------
    dict : résultats par jour (data, centroids, silhouette)
    """
    results = {}
    day_names = [
        "Monday", "Tuesday", "Wednesday", "Thursday",
        "Friday", "Saturday", "Sunday"
    ]

    for day_idx, day_name in enumerate(day_names):
        df_day = dataframe[dataframe["day_of_week"] == day_idx].copy()

        if len(df_day) < 100:
            continue

        if len(df_day) > max_samples:
            df_day = df_day.sample(n=max_samples, random_state=42)

        X_day, X_day_sc, sc_day = get_features(df_day)

        model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        df_day["cluster"] = model.fit_predict(X_day_sc)
        centroids_day = sc_day.inverse_transform(model.cluster_centers_)
        sil_day = silhouette_score(X_day_sc, df_day["cluster"])

        results[day_name] = {
            "data": df_day,
            "centroids": centroids_day,
            "n_clusters": n_clusters,
            "n_points": len(df_day),
            "silhouette": sil_day,
        }

    return results


day_results = cluster_by_day(df_clean, n_clusters=OPTIMAL_K)

print(f"{'Jour':<12} | {'Points':>8} | {'Clusters':>8} | {'Silhouette':>10}")
print("-" * 50)
for day, res in day_results.items():
    print(
        f"{day:<12} | {res['n_points']:>8,} | "
        f"{res['n_clusters']:>8} | {res['silhouette']:>10.4f}"
    )

In [ ]:
for day_name, result in day_results.items():
    data = result["data"]
    centroids = result["centroids"]

    fig = px.scatter_mapbox(
        data, lat="Lat", lon="Lon",
        color=data["cluster"].astype(str),
        zoom=10, height=500,
        title=f"Hot-zones K-Means - {day_name} ({result['n_points']:,} pickups)",
        color_discrete_sequence=px.colors.qualitative.Set2
    )

    if centroids is not None:
        fig.add_trace(go.Scattermapbox(
            lat=centroids[:, 0], lon=centroids[:, 1],
            mode="markers",
            marker=dict(size=12, color="red", symbol="star"),
            name="Centroïdes"
        ))

    fig.update_layout(
        mapbox_style="open-street-map",
        margin=dict(l=0, r=0, t=40, b=0)
    )
    fig.show()

### 10.1 Analyse des patterns temporels

In [ ]:
days = list(day_results.keys())
sils = [day_results[d]["silhouette"] for d in days]
n_pts = [day_results[d]["n_points"] for d in days]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Nombre de pickups par jour", "Silhouette Score par jour")
)

fig.add_trace(
    go.Bar(x=days, y=n_pts, marker_color="#636EFA"), row=1, col=1
)
fig.add_trace(
    go.Bar(x=days, y=sils, marker_color="#EF553B"), row=1, col=2
)

fig.update_layout(height=400, showlegend=False,
                  title_text="Métriques par jour de la semaine")
fig.show()

weekday_keys = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
weekend_keys = ["Saturday", "Sunday"]

avg_wd = np.mean([day_results[d]["n_points"] for d in weekday_keys if d in day_results])
avg_we = np.mean([day_results[d]["n_points"] for d in weekend_keys if d in day_results])
print(f"Pickups moyens en semaine : {avg_wd:,.0f}")
print(f"Pickups moyens le weekend : {avg_we:,.0f}")
print(f"Ratio semaine/weekend     : {avg_wd/avg_we:.2f}x")

In [ ]:
fig = go.Figure()
colors = px.colors.qualitative.Set1

for i, (day_name, result) in enumerate(day_results.items()):
    if result["centroids"] is not None:
        fig.add_trace(go.Scattermapbox(
            lat=result["centroids"][:, 0],
            lon=result["centroids"][:, 1],
            mode="markers",
            marker=dict(size=10, color=colors[i % len(colors)]),
            name=day_name
        ))

fig.update_layout(
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=40.75, lon=-73.97),
        zoom=11
    ),
    height=600,
    title="Tous les centroïdes K-Means par jour de la semaine"
)
fig.show()

print("Les centroïdes sont globalement stables d'un jour à l'autre,")
print("confirmant des hot-zones structurelles à NYC.")
print("Les variations subtiles entre semaine et weekend reflètent")
print("les différences d'usage (travail vs loisirs).")

---
## 11. Recommandations Business <a id="11-recommendations"></a>

### Synthèse des résultats

L'analyse par clustering a permis d'identifier des **hot-zones stables et récurrentes** pour les pickups Uber à New York City. Les principaux constats :

- Les hot-zones sont **concentrées à Manhattan** (Midtown, Financial District, Upper East/West Side)
- Les patterns sont **globalement stables** d'un jour à l'autre, avec des variations subtiles entre semaine et weekend
- K-Means fournit des **centroïdes exploitables** directement comme coordonnées GPS de positionnement

### Recommandations opérationnelles

1. **Positionnement proactif des chauffeurs** : Utiliser les centroïdes K-Means comme points de stationnement recommandés. L'application peut guider les chauffeurs vers la hot-zone la plus proche sous-desservie.

2. **Adaptation jour/heure** : Différencier les recommandations entre semaine (zones d'affaires) et weekend (zones résidentielles/loisirs). L'analyse horaire permettrait un ciblage encore plus fin.

3. **Gestion des zones à faible densité** : Les points identifiés comme bruit par DBSCAN correspondent à des pickups exceptionnels. Ces zones ne justifient pas un positionnement permanent de chauffeurs.

4. **Monitoring continu** : Recalculer les clusters périodiquement pour adapter les recommandations aux évolutions de la demande.

### Limites et pistes d'amélioration

- **Données datées (2014)** : les patterns de mobilité ont évolué (télétravail, nouveaux quartiers)
- **Granularité horaire** : une analyse heure par heure fournirait des recommandations plus précises
- **Variables externes** : intégrer la météo, les événements et les jours fériés
- **Pistes dans le cadre du cours (M06)** : affiner la grille DBSCAN, tester d'autres plages de K, ou la métrique `manhattan` pour les distances ; analyser des créneaux horaires plus fins avec le même pipeline K-Means / DBSCAN
- **Validation terrain** : confronter les clusters à la connaissance métier des chauffeurs

---
## 12. Conclusion et Perspectives <a id="12-conclusion"></a>

Ce projet a permis de :

1. **Justifier le choix de l'apprentissage non-supervisé** pour une problématique sans labels
2. **Concevoir un pipeline complet** de préparation des données géographiques (filtrage, normalisation)
3. **Explorer les données via PCA** pour valider la pertinence du clustering spatial
4. **Appliquer et optimiser K-Means** : coude comme aide visuelle, puis choix du K (parmi **K ≥ 5**) qui maximise la silhouette
5. **Appliquer et optimiser DBSCAN** via grid search sur `eps` et `min_samples`
6. **Démontrer l'amélioration itérative** : les modèles optimisés surpassent les versions naïves sur le Silhouette Score
7. **Comparer objectivement** les deux algorithmes avec des métriques et des visualisations
8. **Généraliser l'analyse** à chaque jour de la semaine
9. **Formuler des recommandations business** actionables pour Uber

Le clustering non-supervisé s'avère un outil puissant pour transformer des données brutes de pickups en **recommandations opérationnelles concrètes** pour les chauffeurs Uber.